# 02 - Race data ingestion and cleaning


## Purpose

This notebook reads the real USA Swimming race-results CSV from `data/csv/usa-swim-race-results.csv`, standardizes event fields, marks best times, and writes prepared race tables to `data/processed`.


## 1) Imports and paths

**Expected output:** The cell should run without errors and make the required packages, paths, or helper tools available for the rest of the notebook.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
CSV_DIR = DATA_DIR / "csv"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RACE_RESULTS_CSV = CSV_DIR / "usa-swim-race-results.csv"
print("Race results CSV:", RACE_RESULTS_CSV)


## 2) Read the raw race CSV

The CSV includes one row per official race result. The `raw_json` column keeps the original USA Swimming fields for traceability.

**Expected output:** Expect a table loaded from CSV. Check the row count, columns, and first/latest rows to confirm the source data is being read correctly.


In [2]:
races_raw = pd.read_csv(RACE_RESULTS_CSV)
print("Rows:", len(races_raw))
print("Columns:", races_raw.columns.tolist())
display(races_raw.head(3))
display(races_raw.tail(3))


Rows: 453
Columns: ['swimmer', 'event', 'course', 'time', 'time_seconds', 'date', 'meet', 'age', 'points', 'team', 'lsc', 'standard', 'rank', 'person_key', 'swim_event_key', 'meet_key', 'sort_key', 'usas_swim_time_key', 'source_file', 'raw_json']


,swimmer,event,course,time,time_seconds,date,meet,age,points,team,lsc,standard,rank,person_key,swim_event_key,meet_key,sort_key,usas_swim_time_key,source_file,raw_json
0,gmo,50 FR SCY,SCY,27.65,27.65,2024-10-26,2024 SN Golden Buoy Tri-Meet,15,456,California Capital Aquatics,SN,BB,NaN,1731452,1,263513,1.010003e+09,170469196,gmo-result,"{""Age"": ""15"", ""Event"": ""50 FR SCY"", ""LSC"": ""SN..."
1,gmo,50 FR SCY,SCY,27.67,27.67,2026-05-09,2026 SN Post High School Meet,16,452,California Capital Aquatics,SN,BB,NaN,1731452,1,282465,1.010003e+09,187739302,gmo-result,"{""Age"": ""16"", ""Event"": ""50 FR SCY"", ""LSC"": ""SN..."
2,gmo,50 FR SCY,SCY,27.92,27.92,2024-12-07,2024 GU Southern Senior Championship - TWST,15,434,California Capital Aquatics,SN,BB,NaN,1731452,1,264566,1.010003e+09,172126872,gmo-result,"{""Age"": ""15"", ""Event"": ""50 FR SCY"", ""LSC"": ""SN..."


,swimmer,event,course,time,time_seconds,date,meet,age,points,team,lsc,standard,rank,person_key,swim_event_key,meet_key,sort_key,usas_swim_time_key,source_file,raw_json
450,gmo,400 IM LCM,LCM,6:07.69,367.69,2024-07-12,2024 SN Long Course Championships ‘The Bill Ro...,15,341,California Capital Aquatics,SN,2020-2024 BB,NaN,1731452,75,258819,1.750037e+09,168417631,gmo-result,"{""Age"": ""15"", ""Event"": ""400 IM LCM"", ""LSC"": ""S..."
451,gmo,400 IM LCM,LCM,6:17.34,377.34,2025-07-18,2025 SN Long Course Championships ‘The Bill Ro...,16,279,California Capital Aquatics,SN,B,NaN,1731452,75,271046,1.750038e+09,179348805,gmo-result,"{""Age"": ""16"", ""Event"": ""400 IM LCM"", ""LSC"": ""S..."
452,gmo,400 IM LCM,LCM,6:17.51,377.51,2025-07-18,2025 SN Long Course Championships ‘The Bill Ro...,16,278,California Capital Aquatics,SN,B,NaN,1731452,75,271046,1.750038e+09,179348816,gmo-result,"{""Age"": ""16"", ""Event"": ""400 IM LCM"", ""LSC"": ""S..."


## 3) Standardize time, date, stroke, and event fields

Downstream notebooks use these stable columns: `race_date`, `event`, `stroke`, `distance`, `course`, `race_time`, and `race_time_sec`.

Event distance is extracted from the event label:

$$
\text{distance} = \text{first number in event name}
$$

Code-style implementation:

```python
races["distance"] = races["event"].apply(parse_event_distance)
```

Race time is stored as both the display time and numeric seconds:

$$
\text{race time seconds} = \text{numeric version of official swim time}
$$

Code-style implementation:

```python
races["race_time_sec"] = pd.to_numeric(races["time_seconds"], errors="coerce")
```

**Expected output:** The cell should define reusable helper logic. The main thing to interpret is that later cells can call the function consistently.


In [3]:
def parse_event_distance(event: str) -> float:
    first = str(event).split()[0]
    try:
        return float(first)
    except ValueError:
        return np.nan


def stroke_from_event(event: str) -> str:
    text = str(event).upper()
    if " FR" in text or "FREE" in text:
        return "freestyle"
    if " FL" in text or "FLY" in text:
        return "butterfly"
    if " BK" in text or "BACK" in text:
        return "backstroke"
    if " BR" in text or "BREAST" in text:
        return "breaststroke"
    if " IM" in text:
        return "individual medley"
    return "other"


def parse_raw_json(value):
    if pd.isna(value):
        return {}
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return {}

races = races_raw.copy()
races["race_date"] = pd.to_datetime(races["date"], errors="coerce")
races["race_time"] = races["time"].astype(str)
races["race_time_sec"] = pd.to_numeric(races["time_seconds"], errors="coerce")
races["distance"] = races["event"].apply(parse_event_distance)
races["stroke"] = races["event"].apply(stroke_from_event)
races["course"] = races["course"].astype(str)
races["age"] = pd.to_numeric(races["age"], errors="coerce")
races["points"] = pd.to_numeric(races["points"], errors="coerce")
races["raw_record"] = races["raw_json"].apply(parse_raw_json)
races = races.dropna(subset=["race_date", "race_time_sec"]).sort_values("race_date").reset_index(drop=True)
display(races.head(3))
display(races.tail(3))


,swimmer,event,course,time,time_seconds,date,meet,age,points,team,...,sort_key,usas_swim_time_key,source_file,raw_json,race_date,race_time,race_time_sec,distance,stroke,raw_record
0,gmo,50 FR SCY,SCY,38.37,38.37,2018-09-16,2018 PC SSF C/B/A+,9,280.0,San Francisco Serpents,...,1.010004e+09,69323752,gmo-result,"{""Age"": ""9"", ""Event"": ""50 FR SCY"", ""LSC"": ""PC""...",2018-09-16,38.37,38.37,50.0,freestyle,"{'Age': '9', 'Event': '50 FR SCY', 'LSC': 'PC'..."
1,gmo,50 BR SCY,SCY,50.68,50.68,2019-02-23,2019 PC VJO C/B/A+,9,274.0,San Francisco Serpents,...,1.140005e+09,94978833,gmo-result,"{""Age"": ""9"", ""Event"": ""50 BR SCY"", ""LSC"": ""PC""...",2019-02-23,50.68,50.68,50.0,breaststroke,"{'Age': '9', 'Event': '50 BR SCY', 'LSC': 'PC'..."
2,gmo,100 BK SCY,SCY,1:31.58,91.58,2019-02-23,2019 PC VJO C/B/A+,9,375.0,San Francisco Serpents,...,1.120009e+09,142314613,gmo-result,"{""Age"": ""9"", ""Event"": ""100 BK SCY"", ""LSC"": ""PC...",2019-02-23,1:31.58,91.58,100.0,backstroke,"{'Age': '9', 'Event': '100 BK SCY', 'LSC': 'PC..."


,swimmer,event,course,time,time_seconds,date,meet,age,points,team,...,sort_key,usas_swim_time_key,source_file,raw_json,race_date,race_time,race_time_sec,distance,stroke,raw_record
450,gmo,200 FR SCY,SCY,2:08.12,128.12,2026-05-09,2026 SN Post High School Meet,16,404.0,California Capital Aquatics,...,1.030013e+09,187739304,gmo-result,"{""Age"": ""16"", ""Event"": ""200 FR SCY"", ""LSC"": ""S...",2026-05-09,2:08.12,128.12,200.0,freestyle,"{'Age': '16', 'Event': '200 FR SCY', 'LSC': 'S..."
451,gmo,50 FR SCY,SCY,27.67,27.67,2026-05-09,2026 SN Post High School Meet,16,452.0,California Capital Aquatics,...,1.010003e+09,187739302,gmo-result,"{""Age"": ""16"", ""Event"": ""50 FR SCY"", ""LSC"": ""SN...",2026-05-09,27.67,27.67,50.0,freestyle,"{'Age': '16', 'Event': '50 FR SCY', 'LSC': 'SN..."
452,gmo,200 BK SCY,SCY,2:27.04,147.04,2026-05-09,2026 SN Post High School Meet,16,408.0,California Capital Aquatics,...,1.130015e+09,187739303,gmo-result,"{""Age"": ""16"", ""Event"": ""200 BK SCY"", ""LSC"": ""S...",2026-05-09,2:27.04,147.04,200.0,backstroke,"{'Age': '16', 'Event': '200 BK SCY', 'LSC': 'S..."


## 4) Mark best times and current-year races

Best time is calculated within each `event` and `course`, so a 50 FR SCY best does not get mixed with another course or event.

For each event-course group:

$$
\text{best time}_{event,course} = \min(\text{race time seconds})
$$

Code-style implementation:

```python
races["event_best_sec"] = races.groupby(["event", "course"])["race_time_sec"].transform("min")
```

Percent off best measures how far a race was from that swimmer's best for the same event and course:

$$
\text{percent off best} =
\frac{\text{race time} - \text{best time}}{\text{best time}} \times 100
$$

Code-style implementation:

```python
races["percent_off_best"] = (
    (races["race_time_sec"] - races["event_best_sec"]) /
    races["event_best_sec"] * 100
)
```

Lower values are better. A value of `0` means the race was a best time.

**Expected output:** Expect paired previews: the first 3 rows show the starting structure, and the latest 3 rows help confirm the most recent records look reasonable.


In [4]:
races["event_course"] = races["event"] + " " + races["course"]
races["event_best_sec"] = races.groupby(["event", "course"])["race_time_sec"].transform("min")
races["is_best_time"] = races["race_time_sec"].eq(races["event_best_sec"])
races["percent_off_best"] = (races["race_time_sec"] - races["event_best_sec"]) / races["event_best_sec"] * 100
latest_year = int(races["race_date"].dt.year.max())
races["is_current_year"] = races["race_date"].dt.year.eq(latest_year)

display(races[["race_date", "event", "course", "race_time", "is_best_time", "percent_off_best", "is_current_year"]].head(3))
display(races[["race_date", "event", "course", "race_time", "is_best_time", "percent_off_best", "is_current_year"]].tail(3))


,race_date,event,course,race_time,is_best_time,percent_off_best,is_current_year
0,2018-09-16,50 FR SCY,SCY,38.37,False,38.770344,False
1,2019-02-23,50 BR SCY,SCY,50.68,False,22.297297,False
2,2019-02-23,100 BK SCY,SCY,1:31.58,False,32.551744,False


,race_date,event,course,race_time,is_best_time,percent_off_best,is_current_year
450,2026-05-09,200 FR SCY,SCY,2:08.12,False,2.775549,True
451,2026-05-09,50 FR SCY,SCY,27.67,False,0.072333,True
452,2026-05-09,200 BK SCY,SCY,2:27.04,False,2.424074,True


## 5) Build prepared summary tables

Best-times, meet, and event summaries compress many race rows into reusable analysis tables.

Meet best-time count:

$$
\text{meet best times} = \sum \mathbf{1}(\text{is best time})
$$

Code-style implementation:

```python
best_times=("is_best_time", "sum")
```

Average meet performance relative to best:

$$
\overline{\text{percent off best}} = \frac{1}{n}\sum_{i=1}^{n}\text{percent off best}_i
$$

Code-style implementation:

```python
avg_percent_off_best=("percent_off_best", "mean")
```

Event summary best time:

$$
\text{event best time} = \min(\text{race time seconds})
$$

Code-style implementation:

```python
best_time_sec=("race_time_sec", "min")
```

**Expected output:** Expect paired previews: the first 3 rows show the starting structure, and the latest 3 rows help confirm the most recent records look reasonable.


In [5]:
best_times = (
    races.sort_values(["event", "course", "race_time_sec"])
    .groupby(["event", "course"], as_index=False)
    .first()
    [["event", "course", "race_date", "race_time", "race_time_sec", "meet", "age", "standard"]]
)

meet_summary = (
    races.groupby(["meet", "race_date"], as_index=False)
    .agg(
        races=("event", "count"),
        best_times=("is_best_time", "sum"),
        avg_percent_off_best=("percent_off_best", "mean"),
        total_points=("points", "sum"),
    )
    .sort_values("race_date")
)

event_summary = (
    races.groupby(["event", "course", "stroke", "distance"], as_index=False)
    .agg(
        swims=("race_time_sec", "count"),
        best_time_sec=("race_time_sec", "min"),
        latest_time_sec=("race_time_sec", "last"),
        first_date=("race_date", "min"),
        latest_date=("race_date", "max"),
    )
)

display(best_times.head(3))
display(best_times.tail(3))
display(meet_summary.tail())
display(event_summary.head(3))
display(event_summary.tail(3))


,event,course,race_date,race_time,race_time_sec,meet,age,standard
0,100 BK LCM,LCM,2025-05-17,1:20.85,80.85,2025 SN CCA Meet,15,B
1,100 BK SCY,SCY,2023-02-05,1:09.09,69.09,2023 SN Senior Swimming Winter Championships,13,2020-2024 BB
2,100 BR LCM,LCM,2024-05-11,1:34.63,94.63,2024 SN CCA IMR/IMX Quad Meet,14,2020-2024 BB


,event,course,race_date,race_time,race_time_sec,meet,age,standard
30,50 FR SCY,SCY,2024-10-26,27.65,27.65,2024 SN Golden Buoy Tri-Meet,15,BB
31,500 FR SCY,SCY,2025-05-07,5:25.90,325.90,2025 SN CIF SAC-JOAQUIN SECTION,15,AA
32,800 FR LCM,LCM,2025-06-08,10:11.04,611.04,2025 SN Summer Sanders LCM Meet,16,A


,meet,race_date,races,best_times,avg_percent_off_best,total_points
158,2026 PN VAST 49th Annual Washington Open,2026-01-17,2,1,3.978822,749.0
159,2026 PN VAST 49th Annual Washington Open,2026-01-18,2,0,8.224598,638.0
162,2026 Squad Quad,2026-04-11,3,0,6.480605,934.0
160,2026 SN CIFSJS Championships,2026-05-06,1,0,2.460878,447.0
161,2026 SN Post High School Meet,2026-05-09,3,0,1.757319,1264.0


,event,course,stroke,distance,swims,best_time_sec,latest_time_sec,first_date,latest_date
0,100 BK LCM,LCM,backstroke,100.0,10,80.85,80.85,2019-06-22,2025-05-17
1,100 BK SCY,SCY,backstroke,100.0,28,69.09,69.10,2019-02-23,2025-05-10
2,100 BR LCM,LCM,breaststroke,100.0,3,94.63,94.63,2023-05-13,2024-05-11


,event,course,stroke,distance,swims,best_time_sec,latest_time_sec,first_date,latest_date
30,50 FR SCY,SCY,freestyle,50.0,45,27.65,27.67,2018-09-16,2026-05-09
31,500 FR SCY,SCY,freestyle,500.0,38,325.90,333.92,2019-12-15,2026-05-06
32,800 FR LCM,LCM,freestyle,800.0,7,611.04,660.88,2022-07-23,2025-07-19


## 6) Save prepared race data

These files are the stable inputs for race analysis notebooks and the web app.

**Expected output:** Expect processed files or folders to be written for later notebooks and web-app views. The key interpretation is that downstream data is now prepared.


In [ ]:
races.to_csv(PROCESSED_DIR / "clean_race_results.csv", index=False)
best_times.to_csv(PROCESSED_DIR / "race_best_times.csv", index=False)
meet_summary.to_csv(PROCESSED_DIR / "race_meet_summary.csv", index=False)
event_summary.to_csv(PROCESSED_DIR / "race_event_summary.csv", index=False)

print("Saved prepared race data:")
print(PROCESSED_DIR / "clean_race_results.csv")
print(PROCESSED_DIR / "race_best_times.csv")
print(PROCESSED_DIR / "race_meet_summary.csv")
print(PROCESSED_DIR / "race_event_summary.csv")
